# Folio: model training and pipeline evaluation

**Synthetic demonstration only.** No real document dataset was supplied.
Perfect scores on generated templates do not establish real-world accuracy or prove
the absence of overfitting. Run cells in order with the project Python environment.
Internet is needed for first-time pretrained model downloads.

This notebook trains a baseline, supports staged MiniLM fine-tuning, and evaluates
independent OCR, metadata and retrieval fixtures.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / 'training').exists():
    ROOT = ROOT.parent
assert (ROOT / 'training').exists(), 'Run from the project or notebooks directory.'
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault('HF_HOME', str(ROOT / '.cache/huggingface'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEED = 42
DATASET = ROOT / 'data/synthetic/documents.csv'
BASELINE_DIR = ROOT / 'artifacts/baseline'
TRANSFORMER_DIR = ROOT / 'artifacts/minilm'
RUN_TRANSFORMER_TRAINING = False  # True: retrain. False: inspect saved transformer run.
print('Project:', ROOT)

Project: d:\Final_Year_Project-Work


## 1. Prepare data and check leakage

Six classes, twelve template families each. Entire groups are assigned to a single split.
Duplicate normalized texts are removed before splitting; augmentation happens only after
splitting and only on training text.

For real data supply a CSV with text, label, group and data_kind columns. Set data_kind
to real; group related people, scans, source documents and template families together.
Exact deduplication cannot detect all near-duplicates: inspect those manually. Reserve
a separate external test set for the final accuracy claim.

In [2]:
from training.data import generate, load_rows, split_rows, audit
if not DATASET.exists():
    generate(DATASET, seed=SEED)
rows = load_rows(DATASET)
splits = split_rows(rows, seed=SEED)
data_audit = audit(rows, splits)
display(pd.DataFrame([
    {'split': name, 'samples': len(items), 'groups': len({r['group'] for r in items})}
    for name, items in splits.items()
]))
display(pd.Series(data_audit['classes'], name='samples_per_class'))
print(data_audit['warning'])

,split,samples,groups
0,train,768,48
1,validation,192,12
2,test,192,12


passport     192
pan          192
aadhaar      192
medical      192
insurance    192
invoice      192
Name: samples_per_class, dtype: int64

Synthetic scores do not estimate performance on real documents. Group related people/templates/scans together.


## 2. Train at two important stages

Stage 1 trains a character TF-IDF classifier on clean text. Stage 2 compares two
regularization strengths with training-only OCR augmentation. The vocabulary uses
training data only. Each epoch records train and validation loss and macro F1.

Early stopping restores the best validation-loss checkpoint. Class weights address
imbalance. The confidence threshold uses validation data, never test data.
The resulting probabilities are not calibrated.

In [3]:
from training.baseline import train
baseline = train(DATASET, BASELINE_DIR, seed=SEED)
display(pd.DataFrame([
    {'stage': c['stage'], 'alpha': c['alpha'], 'epochs': c['epochs_run'],
     'validation_f1': c['validation']['macro_f1'], 'validation_loss': c['validation']['loss']}
    for c in baseline['candidates']
]))
print('Selected:', baseline['selected'])

clean_baseline alpha=0.0001: val F1=1.0000, loss=0.0236
regularized_ocr alpha=0.001: val F1=1.0000, loss=0.1379
regularized_ocr alpha=0.0001: val F1=1.0000, loss=0.0224


,stage,alpha,epochs,validation_f1,validation_loss
0,clean_baseline,0.0001,40,1.0,0.023630
1,regularized_ocr,0.0010,60,1.0,0.137890
2,regularized_ocr,0.0001,7,1.0,0.022417


Selected: {'stage': 'regularized_ocr', 'alpha': 0.0001, 'epochs_run': 7, 'validation': {'accuracy': 1.0, 'macro_f1': 1.0, 'loss': 0.02241660899559575}}


## 3. Inspect learning curves and fitting diagnostics

A widening train/validation gap can indicate overfitting. Low training and validation
performance can indicate underfitting, poor labels or insufficient signal. These
heuristic checks are alerts, not guarantees.

The training function evaluates the held-out test set only after selecting the candidate
and threshold. Do not tune in response to the test scores below. Use validation for
development and an external dataset for final assessment.

In [4]:
history = pd.DataFrame(baseline['history'])
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (stage, alpha), group in history.groupby(['stage', 'alpha']):
    label = f'{stage}, alpha={alpha}'
    axes[0].plot(group.epoch, group.train_loss, '--', label=label+' train')
    axes[0].plot(group.epoch, group.val_loss, label=label+' validation')
    axes[1].plot(group.epoch, group.val_macro_f1, label=label)
axes[0].set(xlabel='Epoch', ylabel='Cross entropy', title='Learning curves')
axes[1].set(xlabel='Epoch', ylabel='Macro F1', title='Validation F1')
for ax in axes: ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print(json.dumps(baseline['diagnostics'], indent=2))
display(pd.DataFrame(baseline['classification_report']).T)
print('Synthetic test:', baseline['test'])

{
  "train_validation_f1_gap": 0.0,
  "possible_overfit": false,
  "possible_underfit": false,
  "note": "Heuristic flags, not proof that underfitting or overfitting is absent."
}


C:\Users\kmbar\AppData\Local\Temp\ipykernel_27136\2799083030.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,precision,recall,f1-score,support
aadhaar,1.0,1.0,1.0,16.0
insurance,1.0,1.0,1.0,32.0
invoice,1.0,1.0,1.0,32.0
medical,1.0,1.0,1.0,48.0
pan,1.0,1.0,1.0,16.0
passport,1.0,1.0,1.0,48.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,192.0
weighted avg,1.0,1.0,1.0,192.0


Synthetic test: {'accuracy': 1.0, 'macro_f1': 1.0, 'loss': 0.02256978241537197, 'group_bootstrap_accuracy_95ci': [1.0, 1.0]}


## 4. Stage the transformer training

MiniLM is the transformer option from the reference architecture. Train its new
classification head first, then unfreeze the encoder with a lower learning rate.
The loop uses dropout, weight decay, class weighting, label smoothing, gradient
clipping, training-only OCR augmentation and best-validation-checkpoint restoration.

Defaults: batch size 4, max sequence length 192, six epochs on a 4 GB GPU.
Long-document truncation is a limitation requiring future chunk-voting evaluation.

Set RUN_TRANSFORMER_TRAINING=True in the first cell to retrain.
Otherwise this cell inspects a saved run; it does not claim a new training run.

In [5]:
if RUN_TRANSFORMER_TRAINING:
    from training.transformer import train as train_transformer
    transformer = train_transformer(DATASET, TRANSFORMER_DIR, epochs=6, batch_size=4, seed=SEED)
    print('Completed transformer training.')
elif (TRANSFORMER_DIR / 'report.json').exists():
    transformer = json.loads((TRANSFORMER_DIR / 'report.json').read_text())
    print('Loaded a previously completed transformer training report.')
else:
    transformer = None
    print('Enable RUN_TRANSFORMER_TRAINING to train MiniLM.')
if transformer:
    display(pd.DataFrame(transformer['history']))
    display(pd.DataFrame(transformer['classification_report']).T)
    print('Diagnostics:', transformer['diagnostics'])

Loaded a previously completed transformer training report.


,epoch,stage,train_loss,train_accuracy,train_macro_f1,val_loss,val_accuracy,val_macro_f1
0,1,head_warmup,1.787267,0.1875,0.052632,1.797475,0.083333,0.025641
1,2,encoder_finetuning,0.161122,1.0000,1.000000,0.166006,1.000000,1.000000
2,3,encoder_finetuning,0.056666,1.0000,1.000000,0.054733,1.000000,1.000000
3,4,encoder_finetuning,0.044952,1.0000,1.000000,0.041425,1.000000,1.000000
4,5,encoder_finetuning,0.042253,1.0000,1.000000,0.037755,1.000000,1.000000
5,6,encoder_finetuning,0.042403,1.0000,1.000000,0.037869,1.000000,1.000000


,precision,recall,f1-score,support
aadhaar,1.0,1.0,1.0,16.0
insurance,1.0,1.0,1.0,32.0
invoice,1.0,1.0,1.0,32.0
medical,1.0,1.0,1.0,48.0
pan,1.0,1.0,1.0,16.0
passport,1.0,1.0,1.0,48.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,192.0
weighted avg,1.0,1.0,1.0,192.0


Diagnostics: {'possible_underfit': False, 'possible_overfit': False, 'train_validation_f1_gap': 0.0}


In [6]:
if transformer:
    h = pd.DataFrame(transformer['history'])
    h.plot(x='epoch', y=['train_loss', 'val_loss'], marker='o', title='MiniLM learning curves')
    plt.ylabel('Cross entropy'); plt.show()
    cm = np.array(transformer['confusion_matrix'])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.imshow(cm, cmap='Blues')
    ax.set(xticks=range(len(transformer['labels'])), yticks=range(len(transformer['labels'])),
           xticklabels=transformer['labels'], yticklabels=transformer['labels'],
           xlabel='Predicted', ylabel='True', title='Synthetic held-out test')
    plt.xticks(rotation=35)
    for i in range(len(cm)):
        for j in range(len(cm)): ax.text(j, i, str(cm[i,j]), ha='center', va='center')
    plt.tight_layout(); plt.show()

C:\Users\kmbar\AppData\Local\Temp\ipykernel_27136\1323347446.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.ylabel('Cross entropy'); plt.show()
C:\Users\kmbar\AppData\Local\Temp\ipykernel_27136\1323347446.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Evaluate downstream stages

Classification accuracy is not end-to-end answer accuracy. Independent synthetic
fixtures test retrieval recall, exact entity matches, missing-evidence abstention
and OCR character/word error rates. OCR uses the installed Tesseract backend;
PaddleOCR is available as a selectable adapter.

Clear rendered text is not representative of photos or handwriting.
Pretrained OCR, speech, translation and LLM components are not retrained without
appropriate labelled data. Evaluate those separately with CER/WER, entity F1,
Recall@k/MRR, answer faithfulness, speech WER, translation chrF/BLEU and TTS listening tests.

In [7]:
from training.evaluate import samples, evaluate
import shutil
samples(ROOT / 'data/samples')
evaluation = evaluate(ROOT / 'artifacts/evaluation', ocr=bool(shutil.which('tesseract')))
display(pd.Series(evaluation['metrics'], name='synthetic_fixture_result'))
print(evaluation['warning'])

retrieval_recall_at_1            1.0
retrieval_recall_at_3            1.0
classification_accuracy          1.0
unrelated_question_abstained    True
labelled_field_exact_match       1.0
ocr_mean_cer                     0.0
ocr_mean_wer                     0.0
Name: synthetic_fixture_result, dtype: object

Twelve rendered OCR fixtures and six document/query fixtures only; not a real-world benchmark. Voice, translation and LLM quality require separate datasets.


## 6. Inference and reproducibility

The app loads a saved model and does not train on user uploads. The default baseline
has low latency. The launch script supports MiniLM with the -Transformer switch.
Only load locally produced model artifacts; pickle/joblib files from others can run code.

In [8]:
from folio.classification import Classifier
classifier = Classifier(BASELINE_DIR / 'model.joblib')
display(classifier.predict('Republic of India Passport. Passport number: P1234567. Expiry date: 2032-06-10.'))
display(pd.DataFrame([
    {'model': 'TF-IDF', **baseline['test']},
    *([{'model':'MiniLM', **transformer['test']}] if transformer else [])
]))
print('Reports:', BASELINE_DIR / 'report.json', TRANSFORMER_DIR / 'report.json')
print('Real-world accuracy remains unmeasured.')

{'label': 'passport',
 'suggested_label': 'passport',
 'confidence': 0.9028556496589413,
 'backend': 'baseline',
 'training_data': 'synthetic',
 'needs_review': False}

,model,accuracy,macro_f1,loss,group_bootstrap_accuracy_95ci
0,TF-IDF,1.0,1.0,0.022570,"[1.0, 1.0]"
1,MiniLM,1.0,1.0,0.037947,NaN


Reports: d:\Final_Year_Project-Work\artifacts\baseline\report.json d:\Final_Year_Project-Work\artifacts\minilm\report.json
Real-world accuracy remains unmeasured.


## 7. Move to real data

1. Obtain permission to use documents and remove unnecessary personal information.
2. Verify class and field labels, including unknown document types.
3. Keep related documents within one split and represent real scan conditions/languages.
4. Tune capacity and regularization using validation only.
5. Freeze the model and thresholds, then evaluate the external test set once.
6. Review per-class recall, calibration, unknown-class rejection and subgroup errors.

See docs/EVALUATION.md for measurement limitations and the real-data workflow.